# Lab 13: AI APIs as Data Tools

*INFO 4617/5617 · Week 13 · Chapter 13, [AI APIs as Data Tools](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html)*

This week every class is live coding: one lab in three parts, one a day. Each step below says what we're building and why, and its code cell holds only the comments that frame it: we write the code live, in class. Nothing here is turned in. After each step, a **💬 Discuss** box holds questions to talk over before the next one; click it to open it.

We'll build one pipeline on the City of Boulder's revenue reports from week 9, the PDFs whose last page is a picture of a table:
- **Part 1, Monday:** **document understanding**, reading a whole PDF, pictures of tables included. [Datalab](https://www.datalab.to)'s API turns it into Markdown a program can read, and we check what it read;
- **Part 2, Wednesday:** **a model as a function**. Through [OpenRouter](https://openrouter.ai), a language model reads one page and returns JSON in a shape we define, and we check its numbers against week 9's;
- **Part 3, Friday:** **search by meaning**. Embeddings, lists of numbers that stand for what a passage is about, find the passages that answer a question. A model answers from those passages and names its sources, or says it can't answer. Last, a look at how MCP and skills describe tools to a model.

No chatbots: every call to a model is one request in and one JSON answer out, and every answer gets checked. Along the way, short ⏱ microsprints ask you to change one thing, alone or in pairs, and share what you find.

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent. You need **two free keys**, each stored as an environment variable by one of week 11's three safe ways (set it, close the terminal, open a new one, and start Jupyter from there):
- `DATALAB_KEY`: sign up at [datalab.to](https://www.datalab.to), then copy a key from [API Keys](https://www.datalab.to/app/keys). The free plan includes a monthly allowance of at least $10 ([pricing](https://www.datalab.to/pricing)); this lab spends about 9 cents of it;
- `OPENROUTER_KEY`: sign up at [openrouter.ai](https://openrouter.ai), then create a key at [Keys](https://openrouter.ai/settings/keys). The models we use end in `:free`: they cost nothing, up to a daily limit.

Treat each key like a password: never paste it into a cell or a code file. On Wednesday and Friday, start at that day's setup step, or run every cell above where we stopped (**Run → Run All Above Selected Cell**).

# Part 1 — Monday: from a PDF to text a program can read

In week 9, pdfplumber read the text layer of Boulder's revenue reports exactly. It stopped at Exhibit 3 of the December 2024 report, a picture of a table, beside a note asking readers to phone the city for "a screen readable version". Today a **document-understanding** API, one that reads a page's pictures as well as its text, reads the whole report. Then we check what it read, the way week 9 checked pdfplumber.

## Step 0 — Set up, and check your keys (5 minutes)

Import what we need, define the User-Agent, and check that both keys are set before reading them, as week 11 did. If one says `MISSING`, quit Jupyter, open a new terminal, and start Jupyter again from there. If you can't restart now, run the given cell: it asks for a missing key with [`getpass()`](https://docs.python.org/3/library/getpass.html), which doesn't show what you type, and does nothing if both keys are set. The chapter's ["Two Keys"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#two-keys) has the steps to get them.

In [ ]:
# Import os, re, time, requests, and pandas (as pd); define HEADERS as in week 12, with your email
# Print whether "DATALAB_KEY" and "OPENROUTER_KEY" are set or MISSING, without printing either key
# Your code here

In [ ]:
# Given: only for a key the check said was MISSING, ask for it with getpass. If both said set, this does nothing
from getpass import getpass
for name in ["DATALAB_KEY", "OPENROUTER_KEY"]:
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name} isn't set. Paste it here: ")

In [ ]:
# Read DATALAB_KEY into datalab_key; datalab_base = "https://www.datalab.to/api/v1"
# book: the address of week 9's saved PDFs, "https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/ch-09/"
# Your code here

<details>
<summary>💬 Discuss</summary>

- Census, FRED and YouTube gave you keys to read their data; Datalab and OpenRouter read whatever you send them. What wouldn't you send?
- Datalab bills by the page, after a free allowance. What does that allowance let the company learn about you?

</details>

## Step 1 — Submit a document (8 minutes)

Datalab's [Convert API](https://documentation.datalab.to/docs/recipes/conversion/conversion-api-overview) takes a PDF, uploaded or by its address, and returns Markdown. We send the address of the book's saved copy of the December 2024 report as `file_url`. We ask for `markdown` in `balanced` mode, and set `paginate` so that each page is marked. Datalab keeps a copy of what it converts, and hands that copy back, free, when the same file comes again. So we also set [`skip_cache`](https://documentation.datalab.to/api-reference/convert-document): Datalab converts the file afresh, and the cost we see is the real one. The key goes in an `X-API-Key` header. The answer isn't the document. Converting takes seconds, so the API answers at once with a **ticket**, `request_check_url`: an address where the result will be when it's ready. See the chapter's ["Submitting a Document"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#submitting-a-document).

In [ ]:
# data: file_url (book + "revenue-report-2024-12.pdf"), output_format "markdown", mode "balanced", paginate "true", skip_cache "true"
# job: the JSON of a POST to f"{datalab_base}/convert" with data= and headers {"X-API-Key": datalab_key} plus HEADERS; print it
# Your code here

<details>
<summary>💬 Discuss</summary>

- Every API so far answered with the data. Why might a service answer with a ticket instead?
- We sent an address, not a file. When would you have to upload the file instead?

</details>

## Step 2 — Wait for the result (10 minutes)

Ask the ticket's address whether the job is done. The [result check](https://documentation.datalab.to/api-reference/convert-result-check) answers with a `status`: `processing`, until it's `complete`. Asking again and again until a job is done is called **polling**. Take it in two moves: ask once by hand and print the status; then a given loop asks every two seconds until the job is complete, the way week 12's loop followed the `cursor` until there wasn't one. The finished answer says how many pages it read, `page_count`, and what they cost, `total_cost`, in cents.

In [ ]:
# check: the JSON of a GET to job["request_check_url"] with the same headers; print its "status"
# Your code here

In [ ]:
# Given: ask again every two seconds until the job is complete; then print what it read and what it cost
while check["status"] != "complete":
    time.sleep(2)
    check = requests.get(job["request_check_url"], headers={"X-API-Key": datalab_key, **HEADERS}).json()
print(check["page_count"], "pages,", check["total_cost"], "cents,", round(check["runtime"], 1), "seconds of processing")

<details>
<summary>💬 Discuss</summary>

- The loop has no limit. If Datalab never finished the job, what would your notebook do?
- Fifteen pages cost a few cents. About what would every year-end report Boulder has published cost?
- When would you choose `lite` mode, which cost the chapter about an eighth as much for this report?

</details>

## Step 3 — Read the Markdown, page by page (12 minutes)

The document is in `check["markdown"]`, one long string. With `paginate`, each page starts with a **page marker**: a number in braces, such as `{14}`, then a line of dashes. In the pattern we split on, `\{\d+\}` matches the number in its braces and `-{10,}` ten or more dashes. So [`re.split()`](https://docs.python.org/3/library/re.html#re.split) on that pattern gives a list of pages; drop the empty piece before the first marker. Python counts from 0, so page 2 is `pages[1]`. Print page 2, the summary table week 9 read with pdfplumber; then page 15, Exhibit 3, the picture of a table. **Don't print page 1:** the cover gives a staff member's email and phone number, and we keep contact details out of our outputs, as week 9 blacked them out. For the same reason, print Exhibit 3 from its table, which starts at the first `|`, below a note with the sales tax office's phone and email. Its lines are long, so print the first 150 characters of each. See the chapter's ["Reading the Markdown"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#reading-the-markdown).

In [ ]:
# pages: re.split(r"\n*\{\d+\}-{10,}\n*", markdown) without its first, empty piece; print how many pages
# Print the first 1,200 characters of pages[1], the summary; then of pages[14], Exhibit 3. Never pages[0]
# Your code here

In [ ]:
# exhibit: pages[14] from its first "|" (the table, below the note); print the first 150 characters of its first 10 lines
# Your code here

### ⏱ Microsprint (4 minutes, alone or in pairs)

Datalab doesn't only read text: where a page has a chart, it writes a description of it. Chart 8, on page 9, is a pie chart of marijuana revenue, with its figures printed on the slices.

**Do:** print page 9 (`pages[8]`), and compare the description under the chart with the figures in its table.
**Share:** one sentence of the description that the figures contradict, and who wrote that sentence: the city, or Datalab?

In [ ]:
# Microsprint: print pages[8]; read the description under Chart 8 against its figures
# Your code here

<details>
<summary>💬 Discuss</summary>

- The city's note asks readers to phone for a screen readable version of Exhibit 3. Who would have to check Datalab's table before the city could publish it as that version?
- The description of Chart 8 calls one slice the largest, then gives a larger one. How would you tell what a model wrote on these pages from what it read off them?
- Page 10's bar chart of sugar-sweetened beverage tax has no figures printed on it. Print `pages[9]`: where could the numbers Datalab wrote beside it have come from?

</details>

## Step 4 — Check what it read (12 minutes)

Week 9's first check was a sum: a row's months should add up to its total ([chapter 9, "Checking an Extraction"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#checking-an-extraction)). Exhibit 3 has 30 rows of figures, one per tax and year. In the Markdown, each row of the table is one line, with its cells between `|` characters. Split each line on `|`, and strip the `<b>` tags and the commas from its cells. Where `cells[2]` is a year, add the twelve months and compare the sum with the total. The report rounds each figure to the dollar, so allow $5, as week 9 did.

In [ ]:
# For each line of pages[14]: cells = its pieces split on "|", stripped, without "<b>", "</b>" and ","
# Keep a line whose cells[2] is a year (re.fullmatch(r"20\d\d", ...)); add the 12 months, cells[3:15], and compare with cells[15]
# Count the rows checked, and print any whose months miss the total by more than $5
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

The 2018 report is older: its Exhibit 3 is a table of text, not a picture.

**Do:** submit `revenue-report-2018.pdf` the same way (Steps 1 and 2), and run Step 4's check on its page 15, `pages_2018[14]`.
**Share:** its page count and cost, how many rows the check found, and how many missed.

In [ ]:
# Microsprint: set data["file_url"] to the 2018 report; submit, wait (the loop again), split into pages_2018, and check pages_2018[14]
# Your code here

<details>
<summary>💬 Discuss</summary>

- Every row added up. What can a sum not catch? (Week 9's Docling put the right figures beside the wrong label.)
- Week 9 ran Docling on a laptop, free, in about a minute; Datalab took seconds and cost cents. What else differs between the two if the PDF holds a student's records rather than a city's?
- Our code assumes every request worked. If Datalab answered 401 (Unauthorized) because the key was wrong, which line would fail first?

</details>

## Step 5 — Save the text for Wednesday (3 minutes)

Write the 2024 report's Markdown to a file, so Wednesday doesn't have to pay to read it again. (Wednesday also works from the book's saved copies of all seven year-end reports, made the same way on 2026-11-07, so a student who missed today can start there.)

In [ ]:
# Write markdown, the 2024 report's, to "revenue-report-2024-12.md" with open(..., "w", encoding="utf-8")
# Your code here

## End of Part 1

On Wednesday, Part 2 hands one page of this Markdown to a language model and asks for JSON in a shape we define: the summary table's totals. Then we check the model's numbers against week 9's, for seven years of reports. It starts with its own setup step, so you don't need today's cells to have run.

# Part 2 — Wednesday: a model as a function

A **language model**, the kind of program behind a chatbot, can be called like any other API: we send it an instruction and one page, and it returns an answer. We won't chat with it. We ask for **structured output**, JSON in a shape we define, so the answer goes straight into a DataFrame. Then we check it, because a model's numbers look right whether or not they are.

## Step 6 — Set up again (5 minutes)

Part 2 starts fresh, so a student who missed Monday can start here. Import what we need, check and read `OPENROUTER_KEY`, and set two addresses: OpenRouter's API, and the book's saved copies of Datalab's Markdown for the seven year-end reports (`data/ch-13/`), made as on Monday. [OpenRouter](https://openrouter.ai/docs/quickstart) takes one kind of request and passes it to models from many companies. Models whose ID ends in `:free` cost nothing, but they're shared, so they're sometimes busy.

In [ ]:
# Import os, re, json, time, requests, and pandas (as pd); define HEADERS; check and read OPENROUTER_KEY
# openrouter_base = "https://openrouter.ai/api/v1"; saved = the book's data/ch-13/ address, as book was Monday's data/ch-09/
# Your code here

In [ ]:
# Given: only for a key the check said was MISSING, ask for it with getpass. If both said set, this does nothing
from getpass import getpass
for name in ["OPENROUTER_KEY"]:
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name} isn't set. Paste it here: ")
openrouter_key = os.environ["OPENROUTER_KEY"]

<details>
<summary>💬 Discuss</summary>

- OpenRouter stands between you and the companies that run the models. What does that make easier?
- With OpenRouter in the middle, who sees each page you send?

</details>

## Step 7 — Find the summary page (5 minutes)

Read the 2024 report's Markdown from the saved copy, split it into pages as on Monday, and keep the first page that holds the table's last row, `Total Sales and Use Tax`. That one page is all the model will see. It's about 1,500 **tokens**. A token is a word or a piece of one: the unit a model reads and writes text in, and is billed by.

In [ ]:
# markdown: the .text of a GET to saved + "revenue-report-2024-12.md"; pages: split as on Monday
# summary: the first page that contains "Total Sales and Use Tax"; print its first 1,000 characters
# Your code here

<details>
<summary>💬 Discuss</summary>

- The phrase is on two pages of every report, and we took the first. What would the model see if that page held a different table?
- What would the model miss if the table ran onto the next page?

</details>

## Step 8 — One request, one JSON answer (15 minutes)

A request to [chat completions](https://openrouter.ai/docs/api/reference/overview) holds a list of `messages`: a `system` message with the instruction, then a `user` message with the page. That's the whole conversation; we never add a third message. `response_format` asks for [structured outputs](https://openrouter.ai/docs/guides/features/structured-outputs). With `type` `json_schema`, the model must answer in the shape of the given `SCHEMA`, a **JSON Schema**: a description, itself in JSON, of the fields an answer must have and the type of each. **Temperature** sets how much chance goes into each word the model picks; `temperature` 0 asks for its most likely answer. We send a **fallback list** of free `models` rather than one model: if the first is busy, OpenRouter tries the next ([model fallbacks](https://openrouter.ai/docs/guides/routing/model-fallbacks)). The answer's text is in `choices[0]["message"]["content"]`, a JSON string, and [`json.loads()`](https://docs.python.org/3/library/json.html#json.loads) turns it into a dictionary. The response's `usage` counts the tokens the request used. Among them are `reasoning_tokens`: text some models write to work an answer out before they give it, which isn't part of the answer. See the chapter's ["One Request, One JSON Object"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#one-request-one-json-object).

In [ ]:
# Given: the shape of the answer, as a JSON Schema: four fields, each with a type and a description
MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]
SCHEMA = {
    "type": "object",
    "properties": {
        "month": {"type": "string", "enum": MONTHS,
                  "description": "The month the year-to-date figures run through"},
        "year": {"type": "integer",
                 "description": "The report's current year: the later of the two years the table compares"},
        "total_prior": {"type": "integer",
                        "description": "The table's last row, Total Sales and Use Tax, for the earlier year, in dollars"},
        "total_current": {"type": "integer",
                          "description": "The table's last row, Total Sales and Use Tax, for the report's current year, in dollars"},
    },
    "required": ["month", "year", "total_prior", "total_current"],
    "additionalProperties": False,
}
INSTRUCTION = ("You extract figures from one page of a City of Boulder revenue report. "
               "Copy numbers exactly as printed, without $ or commas. Use only the page.")

In [ ]:
# MODELS: three free models, in order; body: models, temperature 0, messages (system INSTRUCTION, user summary),
# and response_format {"type": "json_schema", "json_schema": {"name": "totals", "strict": True, "schema": SCHEMA}}
# response: the JSON of a POST to f"{openrouter_base}/chat/completions" with json=body and an Authorization header
# Print the model that answered and its "usage"; answer: json.loads() of the first choice's message content; print it
# Your code here

<details>
<summary>💬 Discuss</summary>

- If a different model answered your neighbor, can you compare your two results?
- Your `usage` shows how many `reasoning_tokens` the model wrote before its answer. Why would they matter on a paid model?
- The free models answer 429, Too Many Requests, when they're busy. If all three did, which line of this cell would fail?

</details>

## Step 9 — Check it against week 9 (10 minutes)

Week 9 read the same table with pdfplumber, by position, and checked its sums, so we trust its figures. They're our **ground truth**: the answers we check others against. Compare each of the model's fields with them: the two totals, the year, and the month. A model that copies numbers well can still misread which year is which.

In [ ]:
# Given: week 9's figures for the last row, Total Sales and Use Tax, read with pdfplumber (chapter 9, "The Time Series"):
# for each report, the year before and the report's own year
TRUTH = {
    "revenue-report-2018": (134538892, 140648710), "revenue-report-2019": (140609751, 141954650),
    "revenue-report-2020": (141954650, 135195413), "revenue-report-2021-12": (135195413, 152302653),
    "revenue-report-2022-12": (152302653, 169112615), "revenue-report-2023-12": (169112615, 175517129),
    "revenue-report-2024-12": (175517129, 173908431),
}

In [ ]:
# prior, current: TRUTH["revenue-report-2024-12"]; print, for each total, the model's figure, week 9's, and whether they match
# Then print the model's year and month: are they 2024 and December?
# Your code here

### ⏱ Microsprint (4 minutes, alone or in pairs)

**Do:** send exactly the same request, `body`, three more times, two seconds apart, and print each answer.
**Share:** did all three agree, with each other and with week 9? Which field changed, if any?

In [ ]:
# Microsprint: send body again three times, sleeping 2 seconds between; print the model and the answer each time
# Your code here

<details>
<summary>💬 Discuss</summary>

- Our three answers agreed with each other, the wrong year included. Does agreement make an answer right?
- `temperature` 0 asks for the most likely answer. Why might your answers still differ from your neighbor's, or from one day to the next?
- If the model had been wrong and you hadn't had week 9's figures, how would you have found out?

</details>

## Step 10 — Seven reports (15 minutes)

Now the same request for every year-end report from 2018 to 2024. We'll send it seven times, so gather Step 8's lines into a function, `extract_totals(page)`, that takes a page and returns the answer dictionary and the model that gave it. Then loop over `TRUTH`'s reports. For each, read its Markdown, find its summary page as in Step 7, call `extract_totals()`, and compare the answer with week 9's. Pause two seconds between requests. See the chapter's ["Seven Reports"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#seven-reports).

In [ ]:
# extract_totals(page): Step 8's body and request, with the page as the user message; return the answer and response["model"]
# Your code here

In [ ]:
# rows: for each report in TRUTH, read its Markdown, find the summary page, extract_totals() it, and add the answer,
# the model, and whether each total matches week 9; sleep 2 seconds between reports. results: a DataFrame; print it
# Your code here

<details>
<summary>💬 Discuss</summary>

- Where on the page did a wrong report's figures come from? (Print its summary page and read the table's last two rows.)
- The 2024 report's year came back as 2025, and a sentence on its page says 2025. Would you fix that in the instruction, the schema's description, or a check afterwards?
- The loop has no error handling. If the fifth request got a 429 from every model in the list, what would `rows` hold when the cell stopped?

</details>

## End of Part 2

On Friday, Part 3 asks questions of all seven reports at once. Embeddings find the passages that answer a question, and a model answers from them, with the IDs of its sources, or says it can't. Then we look at how MCP and skills describe tools to a model. It starts with its own setup step.

# Part 3 — Friday: search by meaning, answer with sources, and tools

Seven reports make about 240 passages. An **embedding** turns a passage into a list of numbers, so that passages about the same thing get similar numbers. A question becomes a list of numbers too, and the passages closest to it are the ones to read. Then a model answers from those passages only, as JSON with the IDs of its sources. Finding the passages is **retrieval**, and writing the answer from them is generation: together, **retrieval-augmented generation**, or **RAG**. Ours keeps the answer to one figure and its sources. Last, a look at how MCP and skills describe tools to a model.

## Step 11 — Set up, and cut the reports into chunks (4 minutes)

Part 3 starts fresh: import what we need (now with [NumPy](https://numpy.org/doc/stable/user/absolute_beginners.html) too), and read `OPENROUTER_KEY`. An embedding model reads at most a few thousand tokens, and one of these pages, a full table, is longer than that. So the given cell cuts each page into **chunks**: pieces of at most about 2,500 characters, broken only between lines. Each chunk gets an ID made of the report, the page, and the chunk's number on that page, such as `2024-12 p.9 #1`. See the chapter's ["Chunks"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#chunks).

In [ ]:
# Import os, re, json, time, requests, numpy (as np), and pandas (as pd); define HEADERS; check and read OPENROUTER_KEY
# openrouter_base and saved, as on Wednesday
# Your code here

In [ ]:
# Given: only for a key the check said was MISSING, ask for it with getpass. If both said set, this does nothing
from getpass import getpass
for name in ["OPENROUTER_KEY"]:
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name} isn't set. Paste it here: ")
openrouter_key = os.environ["OPENROUTER_KEY"]

In [ ]:
# Given: read the seven reports' Markdown, split each into pages, and each page into chunks of at most
# about 2,500 characters, breaking only between lines; each chunk gets an ID such as "2024-12 p.9 #1"
REPORTS = ["revenue-report-2018", "revenue-report-2019", "revenue-report-2020", "revenue-report-2021-12",
           "revenue-report-2022-12", "revenue-report-2023-12", "revenue-report-2024-12"]
chunks = []
for report in REPORTS:
    pages = re.split(r"\n*\{\d+\}-{10,}\n*", requests.get(saved + report + ".md", headers=HEADERS).text)[1:]
    for number, page in enumerate(pages, start=1):
        text, part = "", 1
        for line in page.splitlines(keepends=True):
            if len(text) + len(line) > 2500 and text:
                chunks.append({"id": f"{report[15:]} p.{number} #{part}", "report": report, "text": text})
                text, part = "", part + 1
            text += line
        chunks.append({"id": f"{report[15:]} p.{number} #{part}", "report": report, "text": text})
chunks = pd.DataFrame(chunks)
print(len(chunks), "chunks from", chunks["report"].nunique(), "reports")

## Step 12 — Embed, and search by meaning (10 minutes)

[OpenRouter's embeddings endpoint](https://openrouter.ai/docs/api/reference/embeddings) takes a `model` and an `input`, a list of texts. It returns one **vector** per text: the list of numbers that is the text's embedding. We use one free model, `EMBEDDER`, for every vector. Two models turn the same text into different numbers, so their vectors can't be compared, and an embedding model never gets a fallback list. Embed one chunk by hand first, and count its numbers. Then the given cell embeds all the chunks, 20 to a request, and scales each vector to length 1. (A vector's length here is the square root of its squared numbers added up, not how many numbers it has.) To score a chunk against a question we use **cosine similarity**, which measures how nearly two vectors point the same way: 1 if they point exactly the same way, less the further apart they point. For vectors of length 1, it's their **dot product**: multiply the two lists number by number, and add up. So `vectors @ query` scores every chunk against a question at once.

In [ ]:
# EMBEDDER = "nvidia/nemotron-3-embed-1b:free"; one: the JSON of a POST to f"{openrouter_base}/embeddings"
# with json {"model": EMBEDDER, "input": [chunks["text"][0]]}; print how many numbers its first embedding holds
# Your code here

In [ ]:
# Given: embed every chunk, 20 to a request, with a pause between requests; vectors: one row per chunk
vectors = []
for start in range(0, len(chunks), 20):
    batch = chunks["text"][start:start + 20].tolist()
    response = requests.post(f"{openrouter_base}/embeddings", json={"model": EMBEDDER, "input": batch},
                             headers={"Authorization": f"Bearer {openrouter_key}", **HEADERS}).json()
    vectors += [item["embedding"] for item in response["data"]]
    time.sleep(1)
vectors = np.array(vectors)
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
print(vectors.shape)

In [ ]:
# question: "How much revenue did the state share back from marijuana taxes?"; embed it the same way
# query: its vector, scaled to length 1; scores: vectors @ query; print the five best chunks' IDs and scores
# Your code here

### ⏱ Microsprint (4 minutes, alone or in pairs)

**Do:** ask two questions of your own the same way: one about a topic (a tax, a reason for a change) and one about a figure (a total in a given year).
**Share:** which question found its passage, and which didn't. What do embeddings seem to capture?

In [ ]:
# Microsprint: embed your own question as above, score the chunks, and print the five best
# Your code here

<details>
<summary>💬 Discuss</summary>

- The marijuana question's five best passages came from five different years, none of them 2024. To get 2024's figure, would you reword the question, or keep only 2024's chunks before scoring?
- Why can't you add a second embedding model's vectors to the same table, even for new reports?

</details>

## Step 13 — Answer with sources, as JSON (12 minutes)

Now the generation half of RAG, kept to one figure. We'll search for more than one question, so first gather Step 12's lines into a function, `search(question)`, that embeds a question and returns every chunk's score. Then keep only the 2024 report's chunks, and take the three best for the question as **passages**, each headed by its ID. Send them with the question and a schema whose `value` may be **null**, JSON's word for "no value" (`None` in Python). A null is the model's way to say the passages don't print the answer. Then check its answer against the PDF: week 9's pdfplumber finds $529,220 printed on Chart 8. See the chapter's ["Retrieval-Augmented Extraction"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#retrieval-augmented-extraction).

In [ ]:
# Given: the shape of an answer with its sources. value may be null: the model's way to say the passages don't say
ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "value": {"type": ["integer", "null"],
                  "description": "The figure, in dollars, exactly as printed in a passage; null if no passage prints it"},
        "source_ids": {"type": "array", "items": {"type": "string"},
                       "description": "The IDs of the passages that print the figure; empty if value is null"},
    },
    "required": ["value", "source_ids"],
    "additionalProperties": False,
}
ANSWER_INSTRUCTION = ("Answer the question using only the passages, each headed by its ID. "
                      "Give a figure only if a passage prints it; an estimate read from a chart doesn't count. "
                      "If no passage prints it, answer null.")

In [ ]:
# search(question): Step 12's lines, gathered: embed the question, scale its vector to length 1, and return vectors @ it
# Your code here

In [ ]:
# question: Step 12's marijuana question; chunks["score"]: search(question)
# top: the three best-scoring chunks of "revenue-report-2024-12"; print their IDs; passages: each "ID: text", joined with blank lines
# Your code here

In [ ]:
# body: Wednesday's, with ANSWER_INSTRUCTION, the user message f"Question: {question}\n\n{passages}", and ANSWER_SCHEMA named "answer"
# Send it; print the model that answered and its answer. Is the value 529220, and do its sources include page 9?
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

Page 10's chart of sugar-sweetened beverage tax has no figures printed on it. But beside the chart, the saved Markdown we search has a table Datalab wrote of what the bars seem to show, each figure marked with a `~`.

**Do:** ask "How much sugar-sweetened beverage tax did the city collect in December 2024?" the same way: `search()` it, take 2024's three best chunks, and send them.
**Share:** did the model answer null, or a figure? If a figure, where did it come from?

In [ ]:
# Microsprint: set question; rescore the chunks with search(); take 2024's best three; put them in body's user message, and send it
# Your code here

<details>
<summary>💬 Discuss</summary>

- If the model gave a figure for the beverage tax, it may have copied one of Datalab's estimates. Which check would have caught it?
- The sources let you check an answer. What would you do with an answer whose sources don't print its value?
- The schema's description says `source_ids` is empty when `value` is null; check whether yours was. What can a schema's types enforce that its descriptions can't?

</details>

## Step 14 — Tools: MCP and skills (6 minutes)

Some programs let a model call **tools**, such as a search or a database. The **Model Context Protocol** (MCP) is a shared way for a server to say which tools it offers. You ask it with **JSON-RPC**: a request in JSON that names a `method` for the server to run, with an `id` that its answer repeats. The method `tools/list` returns each tool's name, its description, and an `inputSchema`, a JSON Schema like Wednesday's ([the specification](https://modelcontextprotocol.io/specification/2025-06-18/server/tools)). DeepWiki runs a public MCP server that needs no key. Send it `initialize`, then `tools/list`. It answers as a short **event stream**: lines of text, each starting with a label such as `event:` or `data:`, so the JSON is on the line that starts with `data: `. A **skill** is plainer still: a folder of instructions in Markdown, whose `name` and `description` tell a model when to read the rest. Read the first lines of a public one, for PDFs, and its headings. See the chapter's ["Tools, MCP, and Skills"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#tools-mcp-and-skills).

In [ ]:
# mcp: "https://mcp.deepwiki.com/mcp"; mcp_headers: HEADERS plus "Accept": "application/json, text/event-stream"
# hello: {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-06-18",
#   "capabilities": {}, "clientInfo": {"name": "info4617", "version": "1.0"}}}; POST it; print the first 300 characters of the .text
# Your code here

In [ ]:
# listing: the .text of a POST of {"jsonrpc": "2.0", "id": 2, "method": "tools/list"}; line: its line that starts with "data: "
# For each tool in json.loads() of the line after "data: ", ["result"]["tools"], print its name and its required inputs
# Your code here

In [ ]:
# skill: the .text of a GET to "https://raw.githubusercontent.com/anthropics/skills/main/skills/pdf/SKILL.md"; print its first 5 lines,
# then the headings that start with "### ", the tools it teaches
# Your code here

<details>
<summary>💬 Discuss</summary>

- An MCP tool's `inputSchema` and Wednesday's `response_format` are both JSON Schema. What does each one constrain: what the model may send, or what it must return?
- The PDF skill tells a model to use pypdf and pdfplumber, week 9's tools. If a model followed it on Boulder's reports, which of week 9's checks would still be yours to run?

</details>

## Step 15 — What did we learn? (3 minutes)

In a few sentences, as comments:

- Datalab read a picture of a table and wrote descriptions of charts. Which of its output would you publish, and which would you check first?
- On Wednesday, which reports did the model get wrong, and was the problem the model or what we sent it?
- Step 12's marijuana question found pages from five other years before any of 2024's. Judging by your own questions too, what does an embedding seem to capture? What did keeping only one report's chunks add?
- Every call this week returned JSON, and every answer was checked against something. What would you check your own project's model outputs against?

**Recommended Exercises ahead** (in the chapter's companion notebook, due Sunday):

- Step 1, convert a public PDF you choose, with no private individuals' names: Monday's Steps 1–3.
- Step 2, a schema of your own: Wednesday's `SCHEMA`, with fields your document prints.
- Step 3, extract: Step 8's request, and Step 10's `extract_totals()` for more than one page.
- Step 4, check: Step 4's sums and Step 9's ground truth; find something to check against.
- Step 5, retrieve and answer with sources, including a question your document can't answer: Friday's Steps 11–13.
- Step 6, report the costs: Step 2's `page_count` and `total_cost`, and Step 8's `usage`.

In [ ]:
# Your answers here, as comments


## Friday: your portfolio checkpoint (15 minutes)

The last 15 minutes go to your project. This week's checkpoint, **Analyze your data**, is due Sunday, November 15:

- **An analysis notebook** that runs top to bottom, with its output visible;
- **results that answer at least part of your question**: the counts or comparisons your proposal named;
- **at least one figure** saved in your repository, with a title, labeled axes, and a one-sentence caption;
- **the README's Methods section** and your preliminary **Findings**.

Use the time to run your analysis notebook once, top to bottom, here, where you can ask. If you use a model anywhere in your analysis, say which, when, and what you checked its output against, in Methods.